# Tutorial 1: one disease, three technologies

**ASI-FIMSA Workshop 2026, spatial omics, hands-on**

Every spatial-omics platform makes the same promise: *molecules, with their coordinates*. They
keep it in very different ways, and the differences decide what questions you can ask.

| | Platform | What one measurement is | How many genes |
| --- | --- | --- | --- |
| **1** | **Visium** | a 55 µm spot, a small disc of tissue holding several cells | the whole transcriptome, ~36,600 |
| **2** | **Xenium** | one cell | a targeted panel, 313 |
| **3** | **Atera** | one cell | the whole transcriptome, ~18,000 |

Read the table as a diagonal. Visium gives you every gene but not every cell. Xenium gives you
every cell but only the genes someone chose in advance. Atera, 10x's pre-release
whole-transcriptome chemistry, is the corner that used not to exist.

You will read each dataset with its real reader, meet the `SpatialData` object all three end up
in, and finish by putting the three side by side at matched physical scale. A free Colab session
is all you need: no GPU, a few minutes of compute, ~110 MB of downloads.

> These are **three different specimens** of human invasive breast carcinoma, all FFPE, all from
> 10x Genomics, not three serial sections cut from one block. They are matched by disease and
> preservation, so the differences you will see are the technology rather than the biology.

## 0. Setup

### 0.1 Install

Colab already ships **numpy**, **pandas**, **matplotlib**, **scipy** and **scikit-learn**. What
is missing is the `spatialdata` family, the scverse ecosystem for spatial omics, plus `scanpy`
for the single-cell steps. Every version is pinned to one tested set, because `spatialdata`,
`spatialdata-io` and `spatialdata-plot` are one moving target rather than three.

This takes one to two minutes. If you see a message about restarting the runtime afterwards,
you can ignore it.

In [ ]:
%pip install -q \
    "numpy==2.0.2" \
    "spatialdata==0.8.0" "spatialdata-io==0.7.1" "spatialdata-plot==0.4.1" \
    "scanpy==1.12.3" "igraph==1.0.0" \
    "huggingface_hub==1.28.0" "gdown==6.1.0"

### 0.2 Imports and settings

`spatialdata_plot` looks unused after you import it, and your linter will say so. It is not.
Importing it **registers a `.pl` accessor** onto every `SpatialData` object, which is how
`sdata.pl.render_images(...)` comes to exist. Pandas uses the same trick for `df.plot`.

In [ ]:
import json
import os
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import spatialdata as sd
import spatialdata_io
import spatialdata_plot  # noqa: F401  -- registers the .pl accessor on SpatialData
import zarr

warnings.filterwarnings("ignore")
sc.settings.verbosity = 0
plt.rcParams["figure.dpi"] = 100

print("spatialdata      :", sd.__version__)
print("spatialdata-io   :", spatialdata_io.__version__)
print("spatialdata-plot :", spatialdata_plot.__version__)
print("scanpy           :", sc.__version__)
print("anndata          :", ad.__version__)
print("numpy            :", np.__version__)
print("zarr             :", zarr.__version__)

---

## 1. Visium: 55 µm spots, every gene

The Visium slide is printed with ~5,000 spots in a honeycomb, each **55 µm across** and spaced
**100 µm centre to centre**, carrying millions of copies of a barcode unique to that spot. You stain
and image the section in H&E, then permeabilise it: mRNA diffuses down onto the spot beneath, gets
reverse-transcribed with that spot's barcode attached, and the library is sequenced. The barcode
says which spot a transcript came from, the image says where the spot was.

Because the readout is ordinary RNA-seq you get **the whole transcriptome**, 36,601 genes here.
Because the unit is a 55 µm disc, you do **not** get cells.

We use the public 10x sample **`V1_Breast_Cancer_Block_A_Section_1`**, an invasive ductal carcinoma
with 3,798 spots under tissue, processed with Space Ranger 1.1.0 in 2020.

### 1.1 Download

Two files, ~38 MB:

| File | Size | What it is |
| --- | --- | --- |
| `..._filtered_feature_bc_matrix.h5` | 28 MB | the counts, genes &times; spots |
| `..._spatial.tar.gz` | 10 MB | the H&E images, the spot coordinates and the scale factors |

The same page offers the full-resolution H&E at **1.8 GB**. The tarball already holds a
2000 &times; 2000 downscaled copy, plenty for looking at the slide, so we skip it here. Tutorial 3
does download it, because it trains a model on tiles cut around every spot.

In [ ]:
%%bash
set -euo pipefail
BASE=https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_1
SAMPLE=V1_Breast_Cancer_Block_A_Section_1

mkdir -p visium && cd visium
for f in ${SAMPLE}_filtered_feature_bc_matrix.h5 ${SAMPLE}_spatial.tar.gz; do
    if [ ! -s "$f" ]; then
        echo "downloading $f ..."
        wget -q --tries=5 --timeout=60 --continue "$BASE/$f"
    else
        echo "$f already present, skipping"
    fi
done
tar -xzf ${SAMPLE}_spatial.tar.gz

echo
echo "--- visium/ ---";         ls -lh
echo
echo "--- visium/spatial/ ---"; ls -lh spatial/

### 1.2 Look at that file listing before you go on

In `visium/spatial/` there is a file called **`tissue_positions_list.csv`**: the spot coordinate
table. Space Ranger 1.x (2020) wrote that name with **no header row**; Space Ranger 2.0 and
later writes `tissue_positions.csv` **with** one. Data you download today can be either.

Both `spatialdata_io.visium()` and `scanpy.read_visium()` accept either filename and sniff for a
header, so using a real reader you never notice. Read the CSV by hand and pandas takes the first
*spot* as a header row: you lose one spot, every column is named after a barcode, and nothing
raises. Use the reader.

### 1.3 Read it with the real reader

One line. `spatialdata_io.visium()` finds the counts, the images, the spot table and the scale
factors, checks them against each other, and hands back a `SpatialData` object.

In [ ]:
SAMPLE = "V1_Breast_Cancer_Block_A_Section_1"

visium = spatialdata_io.visium("visium")
visium

### 1.4 The anatomy of a `SpatialData` object

All three platforms end up in an object shaped like the one printed above. It has five kinds of
content, and one idea that ties them together.

| Slot | Holds | Here |
| --- | --- | --- |
| **Images** | raster pictures (H&E, immunofluorescence) | the hi-res and low-res H&E |
| **Labels** | raster segmentation masks, one integer per pixel | *(none for Visium)* |
| **Points** | individual molecules with coordinates | *(none for Visium)* |
| **Shapes** | vector geometry: circles, polygons | the 3,798 spots, as circles |
| **Tables** | measurements as `AnnData`, cells &times; genes | the count matrix |

The idea is **coordinate systems**. Each element keeps its coordinates in whatever units it was
born with, plus a **transformation** into one or more shared systems, which `spatialdata` applies
for you on request. That is why you can overlay a 2000-pixel image on spots recorded against a
20,000-pixel one and have them line up.

In [ ]:
from spatialdata.transformations import get_transformation

# The reader names elements after the sample, so let us give them short handles.
IMG_HIRES = f"{SAMPLE}_hires_image"
SPOTS = SAMPLE
CS_HIRES = f"{SAMPLE}_downscaled_hires"     # the coordinate system the hi-res image is native in

print("--- Images ---")
for name, img in visium.images.items():
    print(f"  {name:52s} {tuple(img.shape)}  (channels, rows, cols)")

print("\n--- Shapes ---")
spots = visium.shapes[SPOTS]
print(f"  {SPOTS:52s} {len(spots):,} {spots.geometry.iloc[0].geom_type} geometries")
print(f"  columns: {list(spots.columns)}   spot radius: {spots['radius'].iloc[0]:.1f} px")

print("\n--- Tables ---")
vis_table = visium.tables["table"]
print(" ", vis_table)

print("\n--- Coordinate systems ---")
for cs in visium.coordinate_systems:
    print(f"  {cs}")
print("\nHow the spots get into the hi-res image's coordinate system:")
print(" ", get_transformation(spots, to_coordinate_system=CS_HIRES))

The transformation printed at the bottom is a plain scale of about 0.0825, the ratio between
the 24,240-pixel full-resolution image and the 2000-pixel one we downloaded. `spatialdata`
carries it so you never have to.

### 1.5 Normalise the counts

Standard scanpy preprocessing. A spot that simply captured more RNA is not "higher" for every
gene, so normalising every spot to the same total and then taking `log1p` puts spots on a
comparable footing. We keep the raw counts in a layer, because raw counts are what you want for
any statistical test later.

In [ ]:
vis_table.layers["counts"] = vis_table.X.copy()      # keep the raw counts

vis_counts_per_spot = np.asarray(vis_table.layers["counts"].sum(axis=1)).ravel()
vis_genes_per_spot = np.asarray((vis_table.layers["counts"] > 0).sum(axis=1)).ravel()
print(f"spots               : {vis_table.n_obs:,}")
print(f"genes               : {vis_table.n_vars:,}")
print(f"median counts / spot: {np.median(vis_counts_per_spot):,.0f}")
print(f"median genes  / spot: {np.median(vis_genes_per_spot):,.0f}")

sc.pp.normalize_total(vis_table, target_sum=1e4)
sc.pp.log1p(vis_table)
print("\nnormalised to 10,000 counts per spot, then log1p")

Roughly 21,000 transcripts and 6,000 distinct genes **per spot**, far more than a single cell
would give, because a spot is not a single cell. That depth is what you are buying with the
resolution you give up.

### 1.6 Put four genes on the tissue

Four markers, chosen so the picture means something to an immunologist:

| Gene | Reads out |
| --- | --- |
| `ERBB2` | HER2, the therapeutic target, amplified in a subset of breast tumours |
| `EPCAM` | epithelium, so: where the tumour cells are |
| `COL1A1` | collagen I, so: fibroblasts and desmoplastic stroma |
| `PTPRC` | CD45, every leukocyte, so: immune infiltration |

`spatialdata-plot` is chained and declarative: each `.pl.render_*` call adds a layer, `.pl.show()`
draws the stack, and a **list** of genes in `color=` gives one panel per gene.

In [ ]:
MARKERS_VIS = ["ERBB2", "EPCAM", "COL1A1", "PTPRC"]
assert all(g in vis_table.var_names for g in MARKERS_VIS)

(
    visium.pl.render_images(IMG_HIRES)
    .pl.render_shapes(SPOTS, color=MARKERS_VIS, cmap="viridis", fill_alpha=0.9)
    .pl.show(coordinate_systems=CS_HIRES, ncols=2, figsize=(11, 9),
             frameon=False, hspace=0.02)
)
plt.show()

Read the four panels against each other. `EPCAM` and `ERBB2` light up the same territory, the
tumour epithelium, while `COL1A1` fills the space *between* those regions. That anticorrelation is
the tumour/stroma architecture of the section, visible without anyone having annotated anything.

Then look at `PTPRC`. It is low and diffuse almost everywhere, with a few brighter patches. An
immunologist looking down a microscope would see obvious lymphoid aggregates. Where are they?

### 1.7 What a 55 µm spot hides

A spot is a disc 55 µm across, about **2,376 µm²**, against a breast epithelial cell of roughly 10 to
20 µm. The usual quoted figure is **1 to 10 cells per spot**; Section 4 computes it from the cell
density the single-cell platforms measure on the same disease.

Two consequences are the reason the other two sections exist. **Every spot is a mixture**: three
T cells among seven tumour cells give a spot that is transcriptionally mostly tumour. And **you
cannot co-localise within a spot**: if a spot has `CD3D` and `CD68`, you cannot tell whether a T cell
is touching a macrophage or whether they are 50 µm apart.

Which sets up the obvious next question: what if the unit of measurement were the cell?

---

## 2. Xenium: single cells, 313 genes

Xenium never sequences anything. It is **imaging**: padlock probes bind their target mRNA *in situ*,
get amplified into a bright rolling-circle product, and the instrument photographs the section over
many rounds of fluorescent readout. Each gene has a codeword, a pattern of on/off across the rounds,
and decoding it gives a transcript with a **sub-micron coordinate**. Cells are segmented from a
nuclear stain plus a boundary stain, and transcripts assigned to whichever cell they fall in.

The cost is that codewords are finite, so somebody chooses the genes in advance. This dataset uses
the 313-gene **Breast Cancer Tumor Microenvironment** panel, on the public
**`Xenium_FFPE_Human_Breast_Cancer_Rep1`** bundle of 167,780 cells.

### 2.1 Download six small files

A full Xenium `outs/` folder is several gigabytes, most of it morphology images and the
per-transcript table. We need six files, ~33 MB, laid out as the reader expects.

| File | Size | What it is |
| --- | --- | --- |
| `experiment.xenium` | 1.4 KB | the run's metadata: pixel size, panel, software version |
| `cells.parquet` | 3.5 MB | one row per cell: centroid, area, transcript counts |
| `cell_feature_matrix.h5` | 12.1 MB | the counts, features &times; cells |
| `cell_boundaries.parquet` | 8.8 MB | the cell outline polygons |
| `nucleus_boundaries.parquet` | 8.3 MB | the nucleus outline polygons |
| `gene_panel.json` | 154 KB | which genes are on the panel, and why |

In [ ]:
%%bash
set -euo pipefail
BASE=https://cf.10xgenomics.com/samples/xenium/1.0.1/Xenium_FFPE_Human_Breast_Cancer_Rep1
PREFIX=Xenium_FFPE_Human_Breast_Cancer_Rep1

mkdir -p xenium && cd xenium
for f in experiment.xenium cells.parquet cell_feature_matrix.h5 \
         cell_boundaries.parquet nucleus_boundaries.parquet gene_panel.json; do
    if [ ! -s "$f" ]; then
        echo "downloading $f ..."
        # note the -O: the reader wants the plain names, not the sample-prefixed ones
        wget -q --tries=5 --timeout=60 -O "$f" "$BASE/${PREFIX}_$f"
    else
        echo "$f already present, skipping"
    fi
done

echo
echo "--- xenium/ ---"; ls -lh

In [ ]:
# What the instrument recorded about this run.
specs = json.loads(Path("xenium/experiment.xenium").read_text())
for key in ["run_name", "preservation_method", "num_cells", "transcripts_per_cell",
            "panel_name", "panel_num_targets_predesigned", "panel_num_targets_custom",
            "pixel_size", "instrument_sn", "analysis_sw_version"]:
    print(f"  {key:32s} {specs[key]}")

### 2.2 One missing file, and a workaround

`spatialdata_io.xenium()` opens **`cells.zarr.zip`** unconditionally, and we did not download it,
because it is **315 MB**. So we make an empty one.

That file carries the raster segmentation masks for bundles from onboard analysis 1.3 and later.
This bundle reports `Xenium-1.0.1`, and on that branch the reader only asks the zarr group two
membership questions, both of which an empty group answers correctly. We switch the masks off
below anyway: `cell_boundaries.parquet` carries the same segmentation as vector polygons, which
is smaller and what we want to draw.

> A Workshop shortcut, not a recipe. For your own Xenium run, download the whole `outs/` folder.

In [ ]:
# Three lines: open a zip-backed zarr store, write one empty group into it, close it.
store = zarr.storage.ZipStore("xenium/cells.zarr.zip", mode="w")
zarr.group(store=store)
store.close()

print("cells.zarr.zip:", Path("xenium/cells.zarr.zip").stat().st_size, "bytes"
      "   (the real one is ~315 MB)")

### 2.3 Read it with the real reader

Same idea as Visium: one call, and everything that follows is `SpatialData`. The keyword
arguments switch off the pieces we did not download: the raster label masks, the per-transcript
table (~1 GB) and the morphology images. `cells_as_circles=True` gives us a cheap circle per
cell, useful for whole-slide views where drawing 167,780 polygons would be wasteful.

In [ ]:
xenium = spatialdata_io.xenium(
    "xenium",
    cells_labels=False,        # raster cell masks    -- needs the real cells.zarr.zip
    nucleus_labels=False,      # raster nucleus masks -- likewise
    transcripts=False,         # transcripts.parquet  -- ~1 GB, not downloaded
    morphology_mip=False,      # morphology images    -- not downloaded
    morphology_focus=False,
    cells_as_circles=True,     # also give us one circle per cell
)
xenium

It worked, and the shape of the object tells the story: **no Images**, because we did not
download any; **three Shapes** elements, because the segmentation arrived as vector geometry;
and one **Table** of 167,780 cells &times; 313 genes.

### 2.4 313 genes, and 228 things that are not genes

The table has 313 columns. The file on disk has 541. The difference is the part of a Xenium run
that most people never look at, and it is the part that tells you whether to believe the other
313.

In [ ]:
# Read the matrix again WITHOUT the gene-expression filter, to see everything in it.
all_features = sc.read_10x_h5("xenium/cell_feature_matrix.h5", gex_only=False)
print(all_features.var["feature_types"].value_counts().to_string())
print(f"\ntotal features in the file : {all_features.n_vars}")
print(f"real genes                 : {xenium.tables['table'].n_vars}")
print("\nexamples:")
for kind in pd.unique(all_features.var["feature_types"]):
    names = all_features.var_names[all_features.var["feature_types"] == kind][:3]
    print(f"  {kind:28s} {', '.join(names)}")

Three families of control, each catching a different way the measurement can lie:

* **Negative control probe** (`NegControlProbe_*`): a real probe against a sequence not in the
  transcriptome. If it lights up, probes are binding where they should not.
* **Negative control codeword** (`NegControlCodeword_*`): a valid codeword never assigned to any
  probe. If it lights up, the decoder is inventing calls out of noise.
* **Blank codeword** (`BLANK_*`): codewords left unused in the codebook, for the same reason.

Together they give a per-cell false-positive rate measured on your own slide. There is no
equivalent in a sequencing-based assay.

In [ ]:
ctrl = all_features[:, all_features.var["feature_types"] != "Gene Expression"].X.sum()
real = all_features[:, all_features.var["feature_types"] == "Gene Expression"].X.sum()
print(f"transcripts called as real genes : {real:,.0f}")
print(f"transcripts called as controls   : {ctrl:,.0f}")
print(f"\ncontrol fraction: {100 * ctrl / (ctrl + real):.2f}%   (well under 1% is healthy)")

del all_features   # 541 x 167,780 is not small; let it go

### 2.5 Coordinate systems, and a unit surprise

Before we crop, print the transformation that maps the cell polygons into the `"global"`
coordinate system.

In [ ]:
from spatialdata.transformations import Identity, get_transformation, set_transformation

cell_polys = xenium.shapes["cell_boundaries"]
print("cell_boundaries -> global :", get_transformation(cell_polys, to_coordinate_system="global"))
print("intrinsic bounds (x0, y0, x1, y1):", np.round(cell_polys.total_bounds, 1))
print("pixel size from experiment.xenium:", specs["pixel_size"], "µm per pixel")
print("\n1 / pixel_size =", round(1 / specs["pixel_size"], 6))

The scale factor is `1 / 0.2125`, so the polygons are stored in **micrometres** while `"global"`
is in **camera pixels**. That is sensible for `spatialdata_io`, because `"global"` is where the
morphology images live and images are indexed in pixels. It is not what we want for a Workshop
about physical scale.

Rather than divide by 0.2125 everywhere and hope, we add a **second coordinate system in
micrometres**. The stored coordinates are already micrometres, so the transformation is the
identity and this is three lines. Everything downstream is then in µm.

In [ ]:
UM = "micrometres"
for name in xenium.shapes:
    set_transformation(xenium.shapes[name], Identity(), to_coordinate_system=UM)

print(xenium.coordinate_systems)

### 2.6 Crop to one square millimetre

167,780 cells is more than we want to cluster in a live session, and a whole slide at cell
resolution is unreadable on a laptop screen anyway. `sdata.query.bounding_box()` cuts a window
out of **every** element at once, images, shapes and table together, which is the whole point
of keeping them in one object.

We take a 1 mm &times; 1 mm square containing a duct and the stroma around it.

In [ ]:
X0, Y0, SIDE = 5500.0, 3000.0, 1000.0      # micrometres

xen_crop = xenium.query.bounding_box(
    axes=("x", "y"),
    min_coordinate=[X0, Y0],
    max_coordinate=[X0 + SIDE, Y0 + SIDE],
    target_coordinate_system=UM,
    filter_table=True,                      # keep only the rows for cells inside the window
)
xen_crop

`cell_boundaries` comes back with slightly more geometries than the table has rows: cells whose
polygon overlaps the window but whose centroid sits outside it. They render as "NA" below, which
is what an edge is.

### 2.7 Cluster it

The standard single-cell recipe: normalise, log, scale so no gene dominates, compress to
principal components, build a k-nearest-neighbour graph, and cut it into communities with
**Leiden**. One Xenium-specific note: we normalise to a target of 100 counts, not 10,000, because
with 313 genes and a median of ~160 transcripts per cell, 10,000 would be a wild extrapolation.

About 20 seconds on ~6,000 cells.

In [ ]:
xen_table = xen_crop.tables["table"]
xen_table.layers["counts"] = xen_table.X.copy()
print(f"cells in the crop: {xen_table.n_obs:,}")

sc.pp.normalize_total(xen_table, target_sum=100)
sc.pp.log1p(xen_table)
xen_table.layers["lognorm"] = xen_table.X.copy()
sc.pp.scale(xen_table, max_value=10)

sc.tl.pca(xen_table, n_comps=30)
sc.pp.neighbors(xen_table, n_neighbors=15, n_pcs=30)
sc.tl.leiden(xen_table, resolution=0.6, flavor="igraph", n_iterations=2, key_added="leiden")

xen_table.X = xen_table.layers["lognorm"]        # put the log-normalised values back for plotting

# Leiden labels are strings, so they sort as 0, 1, 10, 11, 2 ... -- put them back in order.
xen_table.obs["leiden"] = xen_table.obs["leiden"].cat.reorder_categories(
    sorted(xen_table.obs["leiden"].cat.categories, key=int)
)
print(f"\n{xen_table.obs['leiden'].nunique()} clusters")
print(xen_table.obs["leiden"].value_counts().sort_index().to_string())

Clusters are numbers, and numbers are not biology. The next step is to ask what each cluster
expresses, using markers you already trust.

In [ ]:
DOT_MARKERS = {
    "Tumour epithelial": ["EPCAM", "KRT8", "ERBB2", "ESR1"],
    "Proliferating": ["MKI67", "TOP2A"],
    "Myoepithelial": ["KRT14", "KRT5", "ACTA2"],
    "Fibroblast": ["POSTN", "LUM", "PDGFRB"],
    "Endothelial": ["PECAM1", "VWF"],
    "T cell": ["PTPRC", "CD3D", "CD8A", "IL7R"],
    "B / plasma": ["MS4A1", "CD79A", "MZB1"],
    "Myeloid": ["CD68", "LYZ", "ITGAX"],
    "Mast": ["TPSAB1"],
}
sc.pl.dotplot(xen_table, DOT_MARKERS, groupby="leiden", standard_scale="var",
              figsize=(11, 4.5), show=False)
plt.show()

### 2.8 Draw the cell boundaries

One wrinkle first, and it illustrates how `SpatialData` links things. We asked the reader for
`cells_as_circles=True`, so the table declares that it annotates the **`cell_circles`** element.
If we ask to colour `cell_boundaries` by `leiden`, plotting refuses: as far as the object is
concerned, nothing connects that table to those polygons.

`set_table_annotates_spatialelement()` re-points the link. The cell IDs are the same in both, so
the join is valid. We are correcting the bookkeeping, not inventing a relationship.

In [ ]:
xen_table.obs["region"] = pd.Categorical(["cell_boundaries"] * xen_table.n_obs)
xen_crop.set_table_annotates_spatialelement(
    "table", region="cell_boundaries", region_key="region", instance_key="cell_id"
)

(
    xen_crop.pl.render_shapes("cell_boundaries", color="leiden")
    .pl.show(
        coordinate_systems=UM,
        figsize=(8, 7),
        title=f"Xenium — 1 mm² crop, {xen_table.n_obs:,} cells, Leiden clusters",
    )
)
plt.show()

Every one of those outlines is **a cell**, with its own 313-gene expression profile. You can see
the duct wall as a ring of cells one or two thick, the stroma between ducts as elongated cells
with a different cluster identity, and immune cells as small polygons scattered through the
stroma and, in places, pressed against the duct edge.

Ask a question here that Visium could not answer: **are the T cells inside the duct or around
it?** On this picture you answer it by looking. In Section 1 a single 55 µm spot would have
covered the duct wall *and* its neighbouring stroma, and averaged them.

The price is that every cell here has exactly **313 genes**. If the gene you care about is not on
the panel, it does not exist in this dataset. Which brings us to the third technology.

---

## 3. Atera: single cells, ~18,000 genes

**Atera** is 10x's pre-release whole-transcriptome in-situ chemistry: the single-cell resolution of
Xenium, with the panel opened up from a few hundred targets to **18,028**.

It is a *preview*, running on a **Gen2 prototype instrument** as a public demonstration bundle rather
than a released product, so sensitivity, specificity and segmentation are all subject to change.
What it shows you is the next point on the curve.

The vendor bundle is ~65 GB, so two **Staged datasets** were prepared instead:

| Staged dataset | Size | What is in it |
| --- | --- | --- |
| `atera_wholeslide_cells.h5ad` | 19.9 MB | all **170,057** cells: aligned centroids, the vendor's clustering, a UMAP, annotated cell types, and a 69-gene marker panel as the expression matrix. No images. |
| `atera_crop.zarr.zip` | 18.0 MB | the **Crop**: a 2 mm square with H&E at two resolutions, per-cell boundary polygons and the annotated table, in one `SpatialData` object. |

The cell types were assigned from the vendor's clustering by a human reviewing marker evidence
cluster by cluster, not by automatic label transfer. The vocabulary (*Tumour epithelial,
Proliferating tumour, Myoepithelial, Fibroblast, Endothelial, Perivascular, T cell, Dendritic cell,
Macrophage, Plasma cell, Mast, Mixed (plasma+mast), Unassigned*) is that reviewer's.

### 3.1 Fetch the Staged datasets

They live in a public Hugging Face dataset repository, with Google Drive as a mirror.

In [ ]:
HF_REPO = "xiao233333/asi-fimsa-workshop-2026"
GDRIVE_FOLDER = "1ELxQjcswMcO7w4N6Z74u_2Yt3F5UWHbm"
STAGED_DIR = Path("atera")
STAGED_DIR.mkdir(exist_ok=True)


def fetch_staged(filename: str) -> Path:
    '''Fetch one Staged dataset: local override, then Hugging Face, then Google Drive.'''
    out = STAGED_DIR / filename
    if out.exists() and out.stat().st_size > 0:
        print(f"{filename}: already here ({out.stat().st_size / 1e6:.1f} MB)")
        return out

    # 1. Local override -- set WORKSHOP_DATA_DIR to a folder holding the files.
    override = os.environ.get("WORKSHOP_DATA_DIR")
    if override:
        src = Path(override) / filename
        if not src.exists():
            raise FileNotFoundError(f"WORKSHOP_DATA_DIR is set to {override} but {filename} is not in it")
        import shutil
        shutil.copyfile(src, out)
        print(f"{filename}: copied from WORKSHOP_DATA_DIR ({out.stat().st_size / 1e6:.1f} MB)")
        return out

    # 2. Hugging Face (primary).
    try:
        from huggingface_hub import hf_hub_download
        got = hf_hub_download(repo_id=HF_REPO, filename=filename, repo_type="dataset",
                              local_dir=str(STAGED_DIR))
        print(f"{filename}: from Hugging Face ({Path(got).stat().st_size / 1e6:.1f} MB)")
        return Path(got)
    except Exception as hf_error:
        print(f"Hugging Face did not work ({type(hf_error).__name__}); trying the Google Drive mirror.")

    # 3. Google Drive (mirror).
    try:
        import gdown
        gdown.download_folder(id=GDRIVE_FOLDER, output=str(STAGED_DIR), quiet=True, use_cookies=False)
        if out.exists():
            print(f"{filename}: from the Google Drive mirror ({out.stat().st_size / 1e6:.1f} MB)")
            return out
    except Exception as drive_error:
        print(f"The Google Drive mirror did not work either ({type(drive_error).__name__}).")

    print(
        "\n"
        "-------------------------------------------------------------------\n"
        f"Could not fetch '{filename}'.\n"
        "\n"
        "This is a download problem, not a problem with your session, and\n"
        "nothing you have run so far is affected. Please tell a presenter --\n"
        "they have the files on a USB stick.\n"
        "\n"
        "If you would rather try yourself:\n"
        f"  Hugging Face : https://huggingface.co/datasets/{HF_REPO}\n"
        f"  Google Drive : https://drive.google.com/drive/folders/{GDRIVE_FOLDER}\n"
        "Download the file, drag it into the Colab file browser on the left\n"
        "into a folder called 'atera', and re-run this cell.\n"
        "-------------------------------------------------------------------"
    )
    raise RuntimeError(f"could not fetch {filename} -- see the message above")


crop_zip = fetch_staged("atera_crop.zarr.zip")
cells_h5ad = fetch_staged("atera_wholeslide_cells.h5ad")

### 3.2 Open them

`atera_wholeslide_cells.h5ad` is a plain `AnnData`, so `anndata.read_h5ad()` opens it.

The Crop needs one extra step. It is a `SpatialData` object stored as a **zipped** zarr
directory, and `read_zarr()` needs a directory rather than a zip archive, so we unzip first and
then read. The object ships that instruction in its own metadata, which we print below.

In [ ]:
%%bash
set -euo pipefail
cd atera
if [ ! -d atera_crop.zarr ]; then
    unzip -q atera_crop.zarr.zip -d atera_crop.zarr
fi
echo "atera_crop.zarr/ contains:"
ls atera_crop.zarr

In [ ]:
whole = ad.read_h5ad(cells_h5ad)
crop = sd.read_zarr("atera/atera_crop.zarr")

print(whole)
print()
print(crop)
print()
print("what the Crop says about itself:")
print("  ", crop.attrs["atera"]["read_instructions"])

### 3.3 The whole slide, one dot per cell

170,057 cells, coloured by the annotated cell type. Two plotting details matter at this size:
`s=0.3` so a dot is roughly one cell rather than a blob, and `rasterized=True` so the figure is
stored as pixels rather than as 170,057 vector objects, which is the difference between a
notebook that scrolls and one that does not.

In [ ]:
cell_types = list(whole.obs["cell_type"].cat.categories)
palette = dict(zip(cell_types, whole.uns["cell_type_colors"]))
xy_whole = whole.obsm["spatial"]

fig, ax = plt.subplots(figsize=(8.5, 11))
for ct in cell_types:
    keep = (whole.obs["cell_type"] == ct).to_numpy()
    ax.scatter(xy_whole[keep, 0], xy_whole[keep, 1], s=0.3, c=palette[ct],
               linewidths=0, rasterized=True, label=f"{ct}  ({keep.sum():,})")
ax.set_aspect("equal")
ax.invert_yaxis()                       # image convention: y increases downwards
ax.set_xlabel("x (µm)")
ax.set_ylabel("y (µm)")
ax.set_title(f"Atera whole slide — {whole.n_obs:,} cells, {len(cell_types)} annotated types")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
          markerscale=18, fontsize=9)
plt.show()

This is a whole tumour section drawn one cell at a time.

The red masses are ducts filled with tumour epithelium, and at their edges you can pick out the
thin orange line of **myoepithelial** cells that still surrounds many of them: the histological
signature of *ductal carcinoma in situ*. The blue **T cells** are not uniform. In the lower third
of the section they crowd around and between the ducts, while in the upper regions the same ducts
sit in quiet stroma. That is spatially heterogeneous immune infiltration, on one slide.

### 3.4 The same cells without their coordinates

`X_umap` was computed by the vendor pipeline on the full ~18,000-gene matrix, so it is a
whole-transcriptome embedding rather than one derived from the 69 markers we ship.

In [ ]:
umap_xy = whole.obsm["X_umap"]

fig, ax = plt.subplots(figsize=(7.5, 6))
for ct in cell_types:
    keep = (whole.obs["cell_type"] == ct).to_numpy()
    ax.scatter(umap_xy[keep, 0], umap_xy[keep, 1], s=0.3, c=palette[ct],
               linewidths=0, rasterized=True, label=ct)
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
ax.set_title("Atera — expression space")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
          markerscale=18, fontsize=8)
plt.show()

The previous figure and this one contain **the same 170,057 points**. One is arranged by where
the cells were, the other by what they were expressing. Spatial omics is the discipline of
holding both at once, and of noticing when a cluster that looks tidy in UMAP turns out to be two
different places on the slide.

### 3.5 The same markers, at cell resolution

Section 1 put `ERBB2`, `COL1A1` and `PTPRC` on 3,798 Visium spots. Here they are again on 170,057
cells, with `EPCAM` added for contrast, drawn as **detection maps**: every cell in pale grey, and
every cell carrying at least one transcript of that gene in colour.

In [ ]:
GENE_COLOURS = [("EPCAM", "#b2182b"), ("ERBB2", "#762a83"),
                ("PTPRC", "#1f6fb4"), ("COL1A1", "#8c6d3f")]

fig, axes = plt.subplots(1, 4, figsize=(17, 7))
for ax, (gene, colour) in zip(axes, GENE_COLOURS):
    counts = np.asarray(whole[:, gene].X).ravel()
    detected = counts > 0
    ax.scatter(xy_whole[:, 0], xy_whole[:, 1], s=0.3, c="0.87",
               linewidths=0, rasterized=True)                  # every cell, for context
    ax.scatter(xy_whole[detected, 0], xy_whole[detected, 1], s=1.4, c=colour,
               linewidths=0, rasterized=True)                  # cells that detected the gene
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    ax.set_title(f"{gene}\n{detected.sum():,} cells ({100 * detected.mean():.0f}%)", fontsize=11)
fig.suptitle("Cells carrying at least one transcript of each gene", y=1.0)
plt.tight_layout()
plt.show()

Compare these with the Visium panels in Section 1. `EPCAM` fills the ducts, `COL1A1` fills the
space between them, and `PTPRC` concentrates in the lower third of the section and in two or
three dense knots that look like lymphoid aggregates. The Visium `PTPRC` panel was a soft haze
over the whole slide; this one resolves into individual CD45-positive cells you could count,
cluster, or measure the distance from a duct.

### 3.5.1 Now the catch

`EPCAM` was detected in about **47%** of cells but `ERBB2` in about **4%**, and `ERBB2` is a
tumour gene on a slide that is 45% tumour. The tumour cells are expressing it. We are mostly not
*seeing* it, because a whole-transcriptome in-situ assay spreads a fixed budget of detected
transcripts across a far larger number of genes.

In [ ]:
atera_depth = float(np.median(whole.obs["transcript_counts"]))
atera_genes = int(whole.uns["atera"]["n_targets_full_panel"])
xen_depth = float(np.median(xenium.tables["table"].obs["transcript_counts"]))
xen_genes = int(xenium.tables["table"].n_vars)

print(f"Xenium : {xen_depth:>7,.0f} transcripts per cell over {xen_genes:>6,} genes"
      f"   ->  {xen_depth / xen_genes:.3f} counts per gene per cell")
print(f"Atera  : {atera_depth:>7,.0f} transcripts per cell over {atera_genes:>6,} genes"
      f"   ->  {atera_depth / atera_genes:.3f} counts per gene per cell")
print(f"\nper gene, Atera is about "
      f"{(xen_depth / xen_genes) / (atera_depth / atera_genes):.0f}x sparser on this tissue")

Many times more transcripts per cell, spread over nearly sixty times more genes. Per gene the
whole-transcriptome assay comes out several times sparser, and the real gap is wider still: the 313
genes on a Xenium panel were *chosen* to be abundant and informative.

So the summary is not "Xenium, but with every gene". You no longer have to choose the genes, which
removes the biggest limitation of a targeted panel. But any one gene is sparse, so you do the
biology on aggregates, clustering cells and comparing clusters. And this is preview chemistry:
sensitivity is exactly the number you would expect to move most before release.

### 3.6 The Crop: cells on their own H&E

The Crop carries an image: a 2 mm square with the H&E, the per-cell boundary polygons and the table,
in one object and one coordinate system (`"global"`, in **micrometres**).

One word about how this gets drawn. Above a few thousand shapes `spatialdata-plot` switches from
matplotlib to a **`datashader`** backend, which rasterises the geometry onto the canvas instead of
drawing every polygon as a vector. At 16,006 cells across 2 mm you cannot tell the difference. In
the zoom that follows, you very much can.

In [ ]:
crop_table = crop.tables["table"]
print(f"cells in the Crop : {crop_table.n_obs:,}")
print(f"genes shipped     : {crop_table.n_vars}")
print("\nelements:")
for name, img in crop.images.items():
    print(f"  image  {name:12s} {tuple(img.shape)}")
for name, shp in crop.shapes.items():
    print(f"  shapes {name:12s} {len(shp):,} polygons")
print("\ncoordinate systems:", crop.coordinate_systems)

In [ ]:
(
    crop.pl.render_images("he")
    .pl.render_shapes("cell_boundaries", color="cell_type", fill_alpha=0.75,
                      outline_alpha=0)
    .pl.show(coordinate_systems="global", figsize=(10, 8),
             title=f"Atera Crop — 2 mm², H&E with {crop_table.n_obs:,} annotated cells")
)
plt.show()

The pink and purple underneath is a slide any pathologist has been reading for a century. The
colours on top are the molecular identity of every cell in it. The orange rims around the red
duct interiors are myoepithelial cells, identified by `KRT14`, `KRT5` and `ACTA2` rather than by
their shape, and their presence is what makes those ducts *in situ* rather than invasive.

### 3.7 Zoom until you can see nuclei

The Crop ships a second image, `he_zoom`: a 300 µm window at the H&E's **native** 0.2738 µm per
pixel. Same coordinate system, so we simply change the axis limits.

Here we **do** ask for `method="matplotlib"` on the boundaries. A rasterising backend draws at
the resolution of the whole rendered extent, so zooming into a twentieth of it afterwards
magnifies a low-resolution image and the thin outlines smear. **Datashader for the overview,
matplotlib whenever you look closely.**

In [ ]:
zoom = crop.attrs["atera"]["crop_window"]["he_zoom"]
zx, zy, zs = zoom["x0_um"], zoom["y0_um"], zoom["size_um"]

# Count the cells in the window from the table rather than trusting a metadata field.
centroids = crop_table.obsm["spatial"]
in_zoom = ((centroids[:, 0] >= zx) & (centroids[:, 0] < zx + zs)
           & (centroids[:, 1] >= zy) & (centroids[:, 1] < zy + zs))
print(f"he_zoom window: {zs:.0f} µm square at ({zx:.0f}, {zy:.0f}) µm, "
      f"{in_zoom.sum()} cells")

fig, axes = plt.subplots(1, 2, figsize=(13, 6.6))
(
    crop.pl.render_images("he_zoom")
    .pl.show(coordinate_systems="global", ax=axes[0],
             title=f"H&E, {zs:.0f} µm across, native resolution")
)
(
    crop.pl.render_images("he_zoom")
    .pl.render_shapes("cell_boundaries", color="cell_type", fill_alpha=0.35,
                      outline=True, outline_alpha=0.9, outline_color="black",
                      outline_width=0.5, method="matplotlib")
    .pl.show(coordinate_systems="global", ax=axes[1],
             title="the same field, with Atera cells", legend_loc=None)
)
for ax in axes:
    ax.set_xlim(zx, zx + zs)
    ax.set_ylim(zy + zs, zy)            # y downwards, as in the image
plt.tight_layout()
plt.show()

Left panel: nuclei, with the pink collagen band across the top and a scatter of small dark cells
below it. Right panel: the same nuclei, each inside a polygon carrying an expression profile that
spans about eighteen thousand genes.

Look at the band of blue polygons through the middle: T cells, in stroma, adjacent to a duct
whose cells are red. This field is 300 µm on a side. In Section 1 the whole of it would have been
covered by **about a dozen Visium spots**, and the T cells would have shown up as a slightly
raised `PTPRC` value in two or three of them.

---

## 4. The three side by side

Everything below is **computed from the three objects still in memory**. Nothing is quoted.

In [ ]:
def bbox_mm2(xy: np.ndarray) -> float:
    '''Area of the bounding box of a set of coordinates, in mm-squared.'''
    return float(np.ptp(xy[:, 0]) * np.ptp(xy[:, 1]) / 1e6)     # np.ptp, never .ptp() -- numpy 2.0


def equivalent_diameter_um(areas) -> float:
    '''Diameter of a circle with the median area, in micrometres.'''
    return float(np.sqrt(4 * np.median(np.asarray(areas)) / np.pi))


# Visium spot coordinates are in full-resolution pixels; the scale factors give us µm.
scalefactors = json.loads(Path("visium/spatial/scalefactors_json.json").read_text())
VIS_UM_PER_PX = 55.0 / scalefactors["spot_diameter_fullres"]     # a spot IS 55 µm, by design
vis_xy_um = vis_table.obsm["spatial"] * VIS_UM_PER_PX

xen_table_full = xenium.tables["table"]
xen_xy_um = xen_table_full.obsm["spatial"]                       # already µm

comparison = pd.DataFrame(
    [
        {
            "Platform": "Visium",
            "Unit measured": "55 µm spot",
            "Unit size (µm across)": 55.0,
            "Genes the platform measured": vis_table.n_vars,
            "Genes in the object we loaded": vis_table.n_vars,
            "Units on the slide": vis_table.n_obs,
            "Area surveyed (mm²)": bbox_mm2(vis_xy_um),
            "Units per mm²": vis_table.n_obs / bbox_mm2(vis_xy_um),
            "Median counts per unit": float(np.median(vis_counts_per_spot)),
        },
        {
            "Platform": "Xenium",
            "Unit measured": "cell",
            "Unit size (µm across)": equivalent_diameter_um(xen_table_full.obs["cell_area"]),
            "Genes the platform measured": xen_table_full.n_vars,
            "Genes in the object we loaded": xen_table_full.n_vars,
            "Units on the slide": xen_table_full.n_obs,
            "Area surveyed (mm²)": bbox_mm2(xen_xy_um),
            "Units per mm²": xen_table_full.n_obs / bbox_mm2(xen_xy_um),
            "Median counts per unit": float(np.median(xen_table_full.obs["transcript_counts"])),
        },
        {
            "Platform": "Atera",
            "Unit measured": "cell",
            "Unit size (µm across)": equivalent_diameter_um(whole.obs["cell_area"]),
            "Genes the platform measured": int(whole.uns["atera"]["n_targets_full_panel"]),
            "Genes in the object we loaded": whole.n_vars,
            "Units on the slide": whole.n_obs,
            "Area surveyed (mm²)": bbox_mm2(xy_whole),
            "Units per mm²": whole.n_obs / bbox_mm2(xy_whole),
            "Median counts per unit": float(np.median(whole.obs["transcript_counts"])),
        },
    ]
).set_index("Platform")

comparison.round(1)

Three rows of that table are worth saying out loud.

* **Unit size.** 55 µm against roughly 13 µm and 9 µm. In *area* that is a factor of about 17 and
  37: resolution differences look small in a column of numbers and enormous on a slide.
* **Genes.** 36,601, then 313, then 18,028. The 313 is not a smaller version of the 36,601; it is
  a different, hand-chosen 313.
* **Genes we actually loaded.** For Atera this is 69, not 18,028, to keep the Workshop download
  under 40 MB. The platform measured all of them; we are carrying a slice.

### 4.1 How many cells are under a Visium spot?

Section 1 promised to compute this rather than quote it. We have two independent measures of cell
density on the same disease, so we can just multiply.

In [ ]:
SPOT_AREA_UM2 = np.pi * (55 / 2) ** 2
print(f"a Visium spot covers {SPOT_AREA_UM2:,.0f} µm² of tissue\n")

for platform in ["Xenium", "Atera"]:
    density_per_mm2 = comparison.loc[platform, "Units per mm²"]
    cells_per_spot = density_per_mm2 * SPOT_AREA_UM2 / 1e6
    print(f"  at the cell density {platform} measures "
          f"({density_per_mm2:,.0f} cells/mm²): {cells_per_spot:.1f} cells per spot")

print("\n(the two disagree because they segment cells differently -- see the next figure --")
print(" and because a bounding box includes empty slide. The honest answer is 'several'.)")

### 4.2 The same square of tissue, three times

The figure below shows a **500 µm &times; 500 µm** square from each dataset, drawn at the
**same number of micrometres per inch**. No panel is zoomed relative to another. This is what
the three technologies would look like if you laid them on the same bench.

In [ ]:
from matplotlib.patches import Circle

SIDE_UM = 500.0


def window_at(cx: float, cy: float) -> tuple[float, float]:
    return cx - SIDE_UM / 2, cy - SIDE_UM / 2


win_vis = window_at(float(np.median(vis_xy_um[:, 0])), float(np.median(vis_xy_um[:, 1])))
win_xen = window_at(6000.0, 3500.0)          # inside the Xenium crop, on a duct
win_ate = window_at(4250.0, 9400.0)          # inside the Atera Crop, on a duct

FILL, EDGE = "#a8c4de", "#25506e"
fig, axes = plt.subplots(1, 3, figsize=(15, 5.8))

# --- Visium: draw each spot at its true 55 µm diameter -----------------------
ax = axes[0]
inside = ((vis_xy_um[:, 0] >= win_vis[0]) & (vis_xy_um[:, 0] < win_vis[0] + SIDE_UM)
          & (vis_xy_um[:, 1] >= win_vis[1]) & (vis_xy_um[:, 1] < win_vis[1] + SIDE_UM))
for x, y in vis_xy_um[
    (np.abs(vis_xy_um[:, 0] - (win_vis[0] + SIDE_UM / 2)) < SIDE_UM / 2 + 60)
    & (np.abs(vis_xy_um[:, 1] - (win_vis[1] + SIDE_UM / 2)) < SIDE_UM / 2 + 60)
]:
    ax.add_patch(Circle((x, y), 27.5, facecolor=FILL, edgecolor=EDGE, lw=0.9))
ax.set_title(f"Visium\n{int(inside.sum())} spots", fontsize=11)
ax.set_xlim(win_vis[0], win_vis[0] + SIDE_UM)
ax.set_ylim(win_vis[1] + SIDE_UM, win_vis[1])

# --- Xenium and Atera: the real segmentation polygons ------------------------
for ax, polys, win, name in [
    (axes[1], xenium.shapes["cell_boundaries"], win_xen, "Xenium"),
    (axes[2], crop.shapes["cell_boundaries"], win_ate, "Atera"),
]:
    sub = polys.cx[win[0]:win[0] + SIDE_UM, win[1]:win[1] + SIDE_UM]
    sub.plot(ax=ax, facecolor=FILL, edgecolor=EDGE, lw=0.4)
    ax.set_title(f"{name}\n{len(sub):,} cells", fontsize=11)
    ax.set_xlim(win[0], win[0] + SIDE_UM)
    ax.set_ylim(win[1] + SIDE_UM, win[1])

for ax in axes:
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    x_left, y_bottom = ax.get_xlim()[0], ax.get_ylim()[0]
    ax.plot([x_left + 40, x_left + 140], [y_bottom - 45] * 2, lw=3.5, c="k", clip_on=False)
    ax.text(x_left + 90, y_bottom - 60, "100 µm", ha="center", va="bottom", fontsize=9)

fig.suptitle("The same 500 × 500 µm of breast tumour, at matched physical scale", y=1.02)
plt.tight_layout()
plt.show()

Three dozen discs against a couple of thousand cells, at the same scale.

One difference between the two right-hand panels is a segmentation artefact rather than biology.
Xenium's default boundaries are made by **expanding outwards from each nucleus** until neighbours
meet, so they tile the plane with no gaps. The Atera Crop was segmented from a **boundary stain**, so
its polygons follow real membranes and leave extracellular space between them. If you ever compare
cell areas across platforms, check this first.

### 4.3 So which one should you use?

There is no winner, only a question you are trying to answer.

| | **Visium** | **Xenium** | **Atera** |
| --- | --- | --- | --- |
| **Buys you** | every gene, no panel design, deep counts, cheap and mature | true single cells, sub-cellular transcript positions, on-slide error controls | true single cells **and** every gene |
| **Costs you** | resolution: every measurement is a mixture of cells | the panel: an unmeasured gene is unmeasurable, forever | pre-release chemistry |
| **Good for** | discovery when you do not know what to look for; regional questions | testing a hypothesis you can name in genes; cell-cell contact | both at once, when you can get on the instrument |

Two things to take away. Do not use Visium to count immune cells: use it to find the regions where
the immune signal is, then follow up at cell resolution. And deconvolution is not resolution, because
estimating cell-type proportions per spot from a reference is still an estimate of a mixture rather
than a measurement of a cell.

### Where this Tutorial sits

**Tutorial 2** takes the Atera Crop and asks which cells sit next to which. **Tutorial 3** asks what
might be passing between those neighbours, with a ligand-receptor test. **Tutorial 4** goes back to
the Visium sample and asks how much of this you could have predicted from the H&E image alone.